In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn xgboost

In [160]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, accuracy_score,\
confusion_matrix, roc_auc_score, roc_curve

## Import Data

In [161]:
df = pd.read_csv("/content/drive/MyDrive/practice/DelayPrediction/cleaned_data.csv")

In [162]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delay,order_item_id,...,geolocation_state_customer,geolocation_zip_code_prefix_seller,geolocation_lat_seller,geolocation_lng_seller,geolocation_city_seller,geolocation_state_seller,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,1,...,SP,9350.0,-23.680114,-46.452454,maua,SP,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,1,...,BA,31570.0,-19.810119,-43.984727,belo horizonte,MG,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,1,...,GO,14840.0,-21.362358,-48.232976,guariba,SP,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,1,...,RN,31842.0,-19.840168,-43.923299,belo horizonte,MG,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,1,...,SP,8752.0,-23.551707,-46.260979,mogi das cruzes,SP,250.0,51.0,15.0,15.0


In [163]:
df.shape

(109619, 29)

In [164]:
df.describe()

,delay,order_item_id,customer_zip_code_prefix,seller_zip_code_prefix,geolocation_zip_code_prefix_customer,geolocation_lat_customer,geolocation_lng_customer,geolocation_zip_code_prefix_seller,geolocation_lat_seller,geolocation_lng_seller,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000,109619.000000
mean,0.079010,1.198506,35075.926245,24478.563625,35075.926245,-21.246685,-46.216125,24478.563625,-22.801449,-47.240495,2087.938715,30.134457,16.575265,22.975579
std,0.269756,0.707761,29887.814951,27634.610555,29887.814951,5.570509,4.043632,27634.610555,2.703291,2.342634,3736.427270,16.114594,13.419464,11.661143
min,0.000000,1.000000,1003.000000,1001.000000,1003.000000,-36.605374,-72.666706,1001.000000,-36.605374,-64.283946,0.000000,7.000000,2.000000,6.000000
25%,0.000000,1.000000,11090.000000,6429.000000,11090.000000,-23.591202,-48.125580,6429.000000,-23.611243,-48.831547,300.000000,18.000000,8.000000,15.000000
50%,0.000000,1.000000,24241.000000,13575.000000,24241.000000,-22.931016,-46.634516,13575.000000,-23.422313,-46.755211,700.000000,25.000000,13.000000,20.000000
75%,0.000000,1.000000,58755.000000,27930.000000,58755.000000,-20.198222,-43.673679,27930.000000,-21.766477,-46.518082,1800.000000,38.000000,20.000000,30.000000
max,1.000000,21.000000,99980.000000,99730.000000,99980.000000,42.184003,-8.577855,99730.000000,-2.546079,-34.847856,40425.000000,105.000000,105.000000,118.000000


## Feature Extraction

In [165]:
# Display column names
print("Existing columns in the Dataset")
df.columns

Existing columns in the Dataset


Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'delay', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'customer_zip_code_prefix',
       'seller_zip_code_prefix', 'geolocation_zip_code_prefix_customer',
       'geolocation_lat_customer', 'geolocation_lng_customer',
       'geolocation_city_customer', 'geolocation_state_customer',
       'geolocation_zip_code_prefix_seller', 'geolocation_lat_seller',
       'geolocation_lng_seller', 'geolocation_city_seller',
       'geolocation_state_seller', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm'],
      dtype='object')

In [166]:
# Data types of existing columns
df.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object
delay,int64
order_item_id,int64


In [167]:
# Convert order_purchase_timestamp to datetime
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')


In [168]:
# Extract order_month and order_day_of_week
df['order_month'] = df['order_purchase_timestamp'].dt.month
df['order_day_of_week'] = df['order_purchase_timestamp'].dt.dayofweek


In [169]:
!pip install geopy

In [170]:
from geopy.distance import geodesic

# Compute distance_km using latitude & longitude
def calculate_distance(row):
    try:
        customer_loc = (row['geolocation_lat_customer'], row['geolocation_lng_customer'])
        seller_loc = (row['geolocation_lat_seller'], row['geolocation_lng_seller'])
        return geodesic(customer_loc, seller_loc).km
    except:
        return None


In [171]:
df['distance_km'] = df.apply(calculate_distance, axis=1)
df['distance_km']

,distance_km
0,18.051106
1,852.256379
2,511.820721
3,1816.652139
4,30.189028
...,...
109614,474.239469
109615,967.147828
109616,370.231030
109617,370.231030


In [172]:
# Create same_state (1 if customer and seller are in the same state, else 0)
df['same_state'] = (df['geolocation_state_customer'] == df['geolocation_state_seller']).astype(int)


In [173]:
# Select required columns and save to a new dataset
selected_features = ["distance_km", "same_state", "order_day_of_week", "order_month", "delay"]
df[selected_features].to_csv("extracted_features_dataset.csv", index=False)
print("✅ Extracted features saved as 'extracted_features_dataset.csv'")

✅ Extracted features saved as 'extracted_features_dataset.csv'


## Load the new dataset

In [174]:
new_df = pd.read_csv('extracted_features_dataset.csv')
new_df

,distance_km,same_state,order_day_of_week,order_month,delay
0,18.051106,1,0,10,0
1,852.256379,0,1,7,0
2,511.820721,0,2,8,0
3,1816.652139,0,5,11,0
4,30.189028,1,1,2,0
...,...,...,...,...,...
109614,474.239469,1,1,2,0
109615,967.147828,0,6,8,0
109616,370.231030,0,0,1,0
109617,370.231030,0,0,1,0


In [175]:
new_df.columns

Index(['distance_km', 'same_state', 'order_day_of_week', 'order_month',
       'delay'],
      dtype='object')

In [176]:
# Step 2: Handle missing values
# Drop rows with NaN values in essential features (if needed)
new_df = new_df.dropna()

# OR: Fill NaN values with median (recommended for numerical data)
new_df.fillna(new_df.median(), inplace=True)

# Verify if NaNs are handled
print("Missing values after cleaning:\n", new_df.isna().sum())

Missing values after cleaning:
 distance_km          0
same_state           0
order_day_of_week    0
order_month          0
delay                0
dtype: int64


In [177]:
# Step 3: Check class imbalance
print(new_df['delay'].value_counts())

delay
0    100958
1      8661
Name: count, dtype: int64


In [178]:
# Step 4: Handle class imbalance if necessary
majority_class = new_df[new_df.delay == 0]
minority_class = new_df[new_df.delay == 1]


In [179]:
if len(minority_class) / len(df) < 0.2:  # If delay cases are less than 20%
    minority_class = resample(minority_class, replace=True, n_samples=len(majority_class), random_state=42)
    new_df = pd.concat([majority_class, minority_class])


In [180]:
print("Updated Class Distribution:")
print(new_df['delay'].value_counts())


Updated Class Distribution:
delay
0    100958
1    100958
Name: count, dtype: int64


In [181]:
# Step 5: Select features & target
features = ["distance_km", "same_state", "order_day_of_week", "order_month"]
X = new_df[features]
y = new_df["delay"]


In [182]:
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (201916, 4)
y shape: (201916,)


In [183]:
# Step 6: Split into Train, Validation, and Test sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [184]:
# Step 7: Normalize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [185]:
# Step 8: Train multiple models
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": XGBClassifier()
}


In [186]:
best_model = None
best_score = 0

In [187]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    score = accuracy_score(y_val, y_pred)
    print(f"{name} Accuracy: {score}")
    if score > best_score:
        best_score = score
        best_model = model

Logistic Regression Accuracy: 0.5554858520157163
Random Forest Accuracy: 0.9581998877406148
XGBoost Accuracy: 0.7220259517284644


In [188]:
# Step 9: Evaluate on Test Set
y_test_pred = best_model.predict(X_test)
print("Best Model Performance:")
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))

Best Model Performance:
              precision    recall  f1-score   support

           0       1.00      0.91      0.95     15125
           1       0.92      1.00      0.96     15163

    accuracy                           0.96     30288
   macro avg       0.96      0.96      0.96     30288
weighted avg       0.96      0.96      0.96     30288

[[13782  1343]
 [    0 15163]]


In [189]:
!pip joblib

ERROR: unknown command "joblib"


In [190]:
import joblib
# Save the best model
joblib.dump(best_model, "best_order_delay_model.pkl")
print("Best model saved as best_order_delay_model.pkl")

Best model saved as best_order_delay_model.pkl


## Optimization techniques for XGBoost

In [191]:
# Define parameter grid for tuning
# param_grid = {
#     'n_estimators': [100, 200, 300],
#     'max_depth': [3, 5, 7],
#     'learning_rate': [0.01, 0.1, 0.2],
#     'colsample_bytree': [0.3, 0.7, 1.0]
# }

In [192]:
# Initialize XGBoost classifier
# xgb = XGBClassifier()


In [193]:
# Perform Grid Search with cross-validation
# grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid,
#                            scoring='accuracy', cv=5, n_jobs=-1, verbose=2)


In [194]:
# Fit on training data
# grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=None,...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None,
                                     random_state=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.3, 0.7, 1.0],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7],
                         'n_estimators': [100, 200, 300]},
             scoring='accuracy', verbose=2)

In [195]:
# Get best parameters
# best_params = grid_search.best_params_
# print("Best parameters found: ", best_params)

Best parameters found:  {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 300}


In [196]:
# Train optimized model
# optimized_xgb = XGBClassifier(**best_params)
# optimized_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.2, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [197]:
# Evaluate the model
# y_pred = optimized_xgb.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)
# print("Optimized XGBoost Accuracy:", accuracy)

Optimized XGBoost Accuracy: 0.7791864764923402


## Optimization for Random Forest Classifier

In [ ]:

# Define parameter grid for Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Initialize Random Forest classifier
rf = RandomForestClassifier(random_state=42)

# Perform Grid Search with cross-validation
grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf,
                              scoring='accuracy', cv=5, n_jobs=-1, verbose=2)

# Fit on training data
grid_search_rf.fit(X_train, y_train)

# Get best parameters
best_params_rf = grid_search_rf.best_params_
print("Best parameters found for Random Forest: ", best_params_rf)

# Train optimized Random Forest model
optimized_rf = RandomForestClassifier(**best_params_rf, random_state=42)
optimized_rf.fit(X_train, y_train)

# Evaluate the model
y_pred_rf = optimized_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Optimized Random Forest Accuracy:", accuracy_rf)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
